# 05 — Multi-Domain Showcase


> **Note.** This notebook uses the legacy `InfonEngine.query()` path with persona-valence scoring. The cassette-native equivalents live in **[03 — Querying](03_querying.ipynb)**: the `Query` DSL for typed filters, `store.ask()` for calibrated verdicts, and `store.any_of()` for multi-target ranking. The multi-domain claim (same code, any schema) applies to both APIs.

The architecture is domain-agnostic. The same code — SPLADE encoding, anchor projection,
infon extraction, constraint aggregation, persona-based query — works across any domain.
You just define a schema.

This notebook demonstrates three domains side by side:
1. **Geopolitical** — nation-state actions, sanctions, diplomacy
2. **Clinical** — drug approvals, trials, diagnoses
3. **Supply chain** — shipments, disruptions, procurement

In [ ]:
import json
from pathlib import Path
from infon import InfonEngine, InfonConfig

def build_cog(name, schema, documents):
    """Helper: create an infon instance, ingest, return it."""
    path = Path(f"data/{name}_schema.json")
    path.parent.mkdir(exist_ok=True)
    path.write_text(json.dumps(schema, indent=2))
    config = InfonConfig(schema_path=str(path), db_path=f"data/{name}_demo.db")
    cog = InfonEngine(config)
    n = cog.ingest(documents, consolidate_now=True)
    print(f"{name}: {n} infons from {len(documents)} docs, {cog.stats()['constraint_count']} constraints")
    return cog

## Domain 1: Geopolitical

In [ ]:
geo_schema = {
    "us_gov":    {"type": "actor", "tokens": ["united states", "washington", "us"]},
    "china_gov": {"type": "actor", "tokens": ["china", "beijing"]},
    "russia":    {"type": "actor", "tokens": ["russia", "moscow", "kremlin"]},
    "eu":        {"type": "actor", "tokens": ["european union", "eu", "brussels"]},
    "nato":      {"type": "actor", "tokens": ["nato"]},
    
    "sanction":  {"type": "relation", "tokens": ["sanction", "sanctions", "embargo", "restrict"]},
    "negotiate": {"type": "relation", "tokens": ["negotiate", "talks", "diplomacy", "summit"]},
    "deploy":    {"type": "relation", "tokens": ["deploy", "deployment", "station", "mobilize"]},
    "trade":     {"type": "relation", "tokens": ["trade", "tariff", "import", "export"]},
    
    "military":  {"type": "feature", "tokens": ["military", "defense", "armed forces", "troops"]},
    "nuclear":   {"type": "feature", "tokens": ["nuclear", "atomic", "warhead"]},
    "energy":    {"type": "feature", "tokens": ["energy", "oil", "gas", "pipeline"]},
    "cyber":     {"type": "feature", "tokens": ["cyber", "cybersecurity", "hack"]},
    
    "europe":    {"type": "market", "tokens": ["europe", "european"]},
    "pacific":   {"type": "market", "tokens": ["pacific", "indo-pacific", "asia"]},
    "mideast":   {"type": "market", "tokens": ["middle east", "gulf"]},
}

geo_docs = [
    {"id": "g01", "timestamp": "2024-01-10", "text": "The United States imposed new sanctions on Russian energy exports to Europe."},
    {"id": "g02", "timestamp": "2024-02-15", "text": "NATO deployed additional military forces to the Baltic states amid tensions with Russia."},
    {"id": "g03", "timestamp": "2024-03-20", "text": "China and the EU held trade negotiations in Brussels over tariff disputes."},
    {"id": "g04", "timestamp": "2024-04-10", "text": "Russia deployed nuclear-capable missiles near the European border."},
    {"id": "g05", "timestamp": "2024-05-15", "text": "The United States and China resumed diplomatic talks on trade and cybersecurity in the Pacific region."},
    {"id": "g06", "timestamp": "2024-06-20", "text": "The EU sanctioned Russian military leaders over escalating deployments in Europe."},
    {"id": "g07", "timestamp": "2024-07-10", "text": "NATO expanded its military presence in the Pacific with joint exercises."},
    {"id": "g08", "timestamp": "2024-08-15", "text": "China deployed naval forces in the Pacific amid trade tensions with the United States."},
]

geo = build_cog("geopolitical", geo_schema, geo_docs)

In [ ]:
result = geo.query("What sanctions has the US imposed on Russia?", persona="analyst")

print(f"Persona: {result.persona}")
for inf in result.infons[:5]:
    v = result.valence.get(inf.infon_id, 0)
    arrow = "▲" if v > 0.1 else "▼" if v < -0.1 else "─"
    print(f"  {arrow} <<{inf.predicate}, {inf.subject}, {inf.object}>>  conf={inf.confidence:.3f}")
    print(f"    \"{inf.sentence[:80]}\"")
    print()

## Domain 2: Clinical

In [ ]:
clinical_schema = {
    "pfizer":    {"type": "actor", "tokens": ["pfizer"]},
    "moderna":   {"type": "actor", "tokens": ["moderna"]},
    "fda":       {"type": "actor", "tokens": ["fda", "food and drug administration"]},
    "who":       {"type": "actor", "tokens": ["who", "world health organization"]},
    
    "approve":   {"type": "relation", "tokens": ["approve", "approval", "authorize"]},
    "trial":     {"type": "relation", "tokens": ["trial", "study", "phase"]},
    "treat":     {"type": "relation", "tokens": ["treat", "treatment", "therapy"]},
    "develop":   {"type": "relation", "tokens": ["develop", "development", "research"]},
    
    "cancer":    {"type": "feature", "tokens": ["cancer", "oncology", "tumor"]},
    "vaccine":   {"type": "feature", "tokens": ["vaccine", "vaccination", "immunization"]},
    "mrna":      {"type": "feature", "tokens": ["mrna", "messenger rna"]},
    "antibody":  {"type": "feature", "tokens": ["antibody", "antibodies", "immunotherapy"]},
}

clinical_docs = [
    {"id": "c01", "timestamp": "2024-01-15", "text": "Pfizer began Phase 3 trials for its new mRNA cancer vaccine."},
    {"id": "c02", "timestamp": "2024-03-10", "text": "The FDA approved Moderna's updated mRNA vaccine for immunization."},
    {"id": "c03", "timestamp": "2024-05-20", "text": "Pfizer developed a novel antibody treatment for cancer tumors."},
    {"id": "c04", "timestamp": "2024-07-15", "text": "The WHO approved an mRNA vaccine developed by Moderna for global distribution."},
    {"id": "c05", "timestamp": "2024-09-10", "text": "Pfizer's mRNA cancer vaccine showed promising results in Phase 3 trials."},
    {"id": "c06", "timestamp": "2024-11-01", "text": "The FDA approved Pfizer's antibody therapy for cancer treatment."},
]

clinical = build_cog("clinical", clinical_schema, clinical_docs)

In [ ]:
result = clinical.query("What mRNA treatments are in development?", persona="analyst")

print(f"Persona: {result.persona}")
for inf in result.infons[:5]:
    print(f"  <<{inf.predicate}, {inf.subject}, {inf.object}>>  conf={inf.confidence:.3f}")
    print(f"    \"{inf.sentence[:80]}\"")
    print()

print("\nConstraints:")
for c in result.constraints[:5]:
    print(f"  ({c.subject}, {c.predicate}, {c.object})  evidence={c.evidence}  score={c.score:.3f}")

## Domain 3: Supply Chain

In [ ]:
supply_schema = {
    "tsmc":      {"type": "actor", "tokens": ["tsmc", "taiwan semiconductor"]},
    "apple":     {"type": "actor", "tokens": ["apple"]},
    "samsung":   {"type": "actor", "tokens": ["samsung"]},
    "intel":     {"type": "actor", "tokens": ["intel"]},
    
    "ship":      {"type": "relation", "tokens": ["ship", "deliver", "supply", "shipment"]},
    "delay":     {"type": "relation", "tokens": ["delay", "disruption", "shortage", "backlog"]},
    "invest":    {"type": "relation", "tokens": ["invest", "investment", "build", "construct"]},
    "order":     {"type": "relation", "tokens": ["order", "procure", "purchase", "contract"]},
    
    "chip":      {"type": "feature", "tokens": ["chip", "semiconductor", "processor", "wafer"]},
    "memory":    {"type": "feature", "tokens": ["memory", "dram", "nand", "storage"]},
    "ai_chip":   {"type": "feature", "tokens": ["ai chip", "gpu", "accelerator", "ai"]},
    
    "taiwan":    {"type": "market", "tokens": ["taiwan"]},
    "us":        {"type": "market", "tokens": ["us", "united states", "arizona"]},
    "korea":     {"type": "market", "tokens": ["korea", "korean"]},
}

supply_docs = [
    {"id": "s01", "timestamp": "2024-01-10", "text": "TSMC invested $40 billion to build a new semiconductor fabrication plant in Arizona."},
    {"id": "s02", "timestamp": "2024-02-15", "text": "Apple ordered advanced AI chips from TSMC for its next-generation products."},
    {"id": "s03", "timestamp": "2024-04-20", "text": "Samsung faced memory chip production delays at its Korean facilities."},
    {"id": "s04", "timestamp": "2024-06-10", "text": "Intel invested in AI chip development at its US semiconductor plants."},
    {"id": "s05", "timestamp": "2024-08-15", "text": "TSMC shipped advanced AI processors to Apple from its Taiwan facilities."},
    {"id": "s06", "timestamp": "2024-10-01", "text": "Samsung invested in AI chip production to compete with TSMC in the semiconductor market."},
    {"id": "s07", "timestamp": "2024-12-10", "text": "Apple ordered next-generation memory chips from Samsung for delivery to the US."},
]

supply = build_cog("supply_chain", supply_schema, supply_docs)

In [ ]:
result = supply.query("What is TSMC's role in AI chip supply?", persona="analyst")

print(f"Persona: {result.persona}")
for inf in result.infons[:5]:
    print(f"  <<{inf.predicate}, {inf.subject}, {inf.object}>>  conf={inf.confidence:.3f}")
    print(f"    \"{inf.sentence[:80]}\"")
    print()

if result.timeline:
    print("Timeline:")
    for inf in result.timeline[:5]:
        print(f"  {inf.timestamp}  <<{inf.predicate}, {inf.subject}, {inf.object}>>")

## Cross-domain comparison

Each domain uses the exact same code path. The only difference is the schema.

In [ ]:
print(f"{'Domain':15s} {'Anchors':>8s} {'Infons':>8s} {'Constraints':>12s}")
print("-" * 48)
for name, c in [("Geopolitical", geo), ("Clinical", clinical), ("Supply Chain", supply)]:
    s = c.stats()
    print(f"{name:15s} {s['anchors']:8d} {s['infon_count']:8d} {s['constraint_count']:12d}")

In [ ]:
geo.close()
clinical.close()
supply.close()
print("Done.")

---

**The pattern:**

1. Define anchor schema for your domain
2. Ingest documents
3. Query with natural language + persona

No training. No model changes. Just schema design.

**Next:** [06 Agent Tools](06_agent_tools.ipynb) — wire infon into an LLM agent.